# Train a voice model for song-cloner

Trains an RVC voice model from recordings **you provide** and saves it to your Google Drive
(and, optionally, to a private Hugging Face repo so Claude cloud sessions can fetch it).

Before you start:

1. **Runtime → Change runtime type → GPU** (a T4 is enough). Training on CPU is impractically slow.
2. Put the recordings of one singer in a Google Drive folder. Aim for 10–30 minutes of audio in total.
   Clean solo vocals work best; if you only have full songs, set `ISOLATE_VOCALS = True` below and the
   music, backing vocals and reverb are stripped first.

The model you train is for your own listening. Do not publish it or covers made with it.

In [ ]:
# --- Settings: edit these ---------------------------------------------------
VOICE_NAME = "myvoice"                           # letters, digits, - _ .
DATASET_DIR = "/content/drive/MyDrive/song-cloner/dataset"   # folder of recordings
ISOLATE_VOCALS = True                            # True if the recordings are full songs
EPOCHS = 200                                     # 150-300 is typical for 10-30 minutes of audio
BATCH_SIZE = 8                                   # lower to 4 if the GPU runs out of memory
SAVE_DIR = "/content/drive/MyDrive/song-cloner/voices"       # where the finished model is copied
HF_REPO = ""                                     # optional, e.g. "your-username/song-cloner-voices"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Install song-cloner with its engine (takes a few minutes).
!apt-get -qq install -y sox > /dev/null
!pip install -q "song-cloner[engine] @ git+https://github.com/aniryou/song-cloner"

In [ ]:
import os
# Keep engine data on the fast local disk; only the finished model goes to Drive.
os.environ["SONGCLONER_HOME"] = "/content/songcloner-data"
os.environ["SONGCLONER_OUTPUTS"] = "/content/songcloner-outputs"
!songcloner doctor

The `Engine:` line above must say **GPU**. If it says CPU, change the runtime type and start again.

In [ ]:
import shlex, subprocess
command = [
    "songcloner", "train", VOICE_NAME, DATASET_DIR,
    "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--force",
]
if ISOLATE_VOCALS:
    command.append("--isolate-vocals")
print(shlex.join(command))
subprocess.run(command, check=True)

In [ ]:
# Copy the finished model to Google Drive.
import shutil
from pathlib import Path
source = Path(os.environ["SONGCLONER_HOME"]) / "models" / "rvc" / "voice_models" / VOICE_NAME
target = Path(SAVE_DIR) / VOICE_NAME
target.mkdir(parents=True, exist_ok=True)
for file in source.iterdir():
    if file.suffix in {".pth", ".index"}:
        shutil.copy2(file, target / file.name)
        print("saved", target / file.name)

## Optional: make the model available to Claude cloud sessions

Cloud sessions start from an empty machine, so the model has to be downloaded each time. A **private**
Hugging Face model repo is the simplest place to keep it:

1. Create a token with *write* access at huggingface.co → Settings → Access Tokens.
2. In Colab, open the key icon (Secrets) in the left bar, add a secret named `HF_TOKEN` with that token,
   and allow this notebook to read it.
3. Set `HF_REPO` in the settings cell and run the cell below. The repo is created private, and the
   upload is refused if the repo turns out to be public.

In [ ]:
if HF_REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    subprocess.run(["songcloner", "voices", "push", VOICE_NAME, "--repo", HF_REPO], check=True)
else:
    print("HF_REPO is empty: skipped. The model is on your Google Drive.")

## Optional: try the voice on a song right here

In [ ]:
SONG = "/content/drive/MyDrive/song-cloner/songs/example.mp3"   # a song file on your Drive
if Path(SONG).is_file():
    subprocess.run(["songcloner", "convert", SONG, "--voice", VOICE_NAME], check=True)
    for file in sorted(Path(os.environ["SONGCLONER_OUTPUTS"]).iterdir()):
        shutil.copy2(file, Path(SAVE_DIR).parent / file.name)
        print("saved", Path(SAVE_DIR).parent / file.name)
else:
    print("Set SONG to an audio file on your Drive to try a conversion.")